# Full Pipeline Demo
This notebook runs the complete PhysNet-HGV tracking pipeline, integrating multi-sensor fusion, PINN predictions, and UKF tracking.

In [ ]:
import numpy as np
import torch
from simulation.trajectory_gen import TrajectoryGenerator
from preprocessing.blackout_labeler import BlackoutLabeler
from filters.ukf_tracker import UKFTracker
from models.pinn_module import PINNModule
from agents.sensor_router import SensorOrchestrationAgent

In [ ]:
# Initialize Components
gen = TrajectoryGenerator()
labeler = BlackoutLabeler(blackout_threshold=1e18)
ukf = UKFTracker(dt=0.1)
pinn = PINNModule(state_dim=6, hidden_dim=256, n_layers=6)
agent = SensorOrchestrationAgent()

In [ ]:
# Generate Trajectory
traj = gen.generate_batch(1)[0]
true_traj = traj['trajectory']
meas_traj = traj['radar_returns']
mask = labeler.label_trajectory(traj['plasma_profile'])

In [ ]:
# Multi-sensor Agent Fusion Simulation
print("Running multi-sensor orchestration agent...")
for i, m in enumerate(mask):
    env_state = {
        'radar_avail': not m,
        'ir_avail': True,
        'optical_avail': True,
        'measurement': meas_traj[i].tolist() if not np.isnan(meas_traj[i]).any() else [0.0]*6
    }
    fused_meas, weights = agent.run(env_state)
print("Multi-sensor fusion complete.")

In [ ]:
# Run Tracker with PINN assistance
print("Running UKF Tracker with PINN...")
estimates, covariances = ukf.run_filter(meas_traj, mask, pinn_model=pinn)
print("Tracking complete.")